In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

!pip install -q ultralytics

import os
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.nn.modules import C3k2, SPPF, Conv, Concat, Detect
from ultralytics.utils.loss import v8DetectionLoss
import ultralytics.utils.metrics
import ultralytics.nn.tasks
import ultralytics.nn.modules as ul_modules
from copy import deepcopy


class EfficientViT_R8(nn.Module):
    """
    Memory-efficient EfficientViT R8 Attention.
    Uses linear attention instead of full quadratic attention.
    """
    def __init__(self, c1, c2=None, num_heads=8, *args, **kwargs):
        super().__init__()
        c2 = c2 if c2 is not None else c1
        self.c1 = c1
        self.c2 = c2
        
        self.num_heads = min(num_heads, c1)
        while c1 % self.num_heads != 0 and self.num_heads > 1:
            self.num_heads -= 1
        
        self.head_dim = c1 // self.num_heads
        self.scale = self.head_dim ** -0.5
        
        self.dw_conv = nn.Conv2d(c1, c1, 3, 1, 1, groups=c1, bias=False)
        self.bn_dw = nn.BatchNorm2d(c1)
        self.q_proj = nn.Conv2d(c1, c1, 1, bias=False)
        self.k_proj = nn.Conv2d(c1, c1, 1, bias=False)
        self.v_proj = nn.Conv2d(c1, c1, 1, bias=False)
        
        self.proj = nn.Conv2d(c1, c2, 1, bias=False)
        self.bn_proj = nn.BatchNorm2d(c2)
        
        # Local feature enhancement (main feature extraction path)
        self.local_conv = nn.Sequential(
            nn.Conv2d(c1, c1, 3, 1, 1, groups=c1, bias=False),
            nn.BatchNorm2d(c1),
            nn.SiLU(inplace=True),
            nn.Conv2d(c1, c2, 1, bias=False),
            nn.BatchNorm2d(c2)
        )
        
        self.residual_conv = nn.Identity() if c1 == c2 else nn.Sequential(
            nn.Conv2d(c1, c2, 1, bias=False),
            nn.BatchNorm2d(c2)
        )
        
        self.act = nn.SiLU(inplace=True)
        
        self.window_size = 8

    def _window_attention(self, x):
        """
        Perform windowed attention to reduce memory usage.
        Splits feature map into windows and applies attention within each window.
        """
        B, C, H, W = x.shape
        ws = self.window_size
        
        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        
        _, _, Hp, Wp = x.shape
        
        x_dw = self.bn_dw(self.dw_conv(x))
        q = self.q_proj(x_dw)
        k = self.k_proj(x_dw)
        v = self.v_proj(x_dw)
        
        num_h, num_w = Hp // ws, Wp // ws
        
        q = q.view(B, C, num_h, ws, num_w, ws).permute(0, 2, 4, 1, 3, 5)
        k = k.view(B, C, num_h, ws, num_w, ws).permute(0, 2, 4, 1, 3, 5)
        v = v.view(B, C, num_h, ws, num_w, ws).permute(0, 2, 4, 1, 3, 5)
        q = q.reshape(B * num_h * num_w, self.num_heads, self.head_dim, ws * ws).transpose(-2, -1)
        k = k.reshape(B * num_h * num_w, self.num_heads, self.head_dim, ws * ws).transpose(-2, -1)
        v = v.reshape(B * num_h * num_w, self.num_heads, self.head_dim, ws * ws).transpose(-2, -1)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = attn @ v  
        
        out = out.transpose(-2, -1).reshape(B, num_h, num_w, C, ws, ws)
        out = out.permute(0, 3, 1, 4, 2, 5).reshape(B, C, Hp, Wp)
        if pad_h > 0 or pad_w > 0:
            out = out[:, :, :H, :W]
        
        return out

    def _linear_attention(self, x):
        """
        Linear attention O(N) instead of O(N^2).
        Uses kernel feature map approximation.
        """
        B, C, H, W = x.shape
        N = H * W
        
        x_dw = self.bn_dw(self.dw_conv(x))
        q = self.q_proj(x_dw).view(B, self.num_heads, self.head_dim, N)
        k = self.k_proj(x_dw).view(B, self.num_heads, self.head_dim, N)
        v = self.v_proj(x_dw).view(B, self.num_heads, self.head_dim, N)
        q = F.elu(q) + 1
        k = F.elu(k) + 1
        
        kv = torch.einsum('bhdn,bhen->bhde', k, v)  
        qkv = torch.einsum('bhdn,bhde->bhen', q, kv) 
        
        k_sum = k.sum(dim=-1, keepdim=True) 
        qk_sum = torch.einsum('bhdn,bhdk->bhn', q, k_sum).unsqueeze(2)  
        out = qkv / (qk_sum + 1e-6)
        
        out = out.reshape(B, C, H, W)
        return out

    def forward(self, x):
        B, C, H, W = x.shape
        residual = self.residual_conv(x)
        
        if H * W <= 256: 
            attn_out = self._linear_attention(x)
        else:
            attn_out = self._window_attention(x)
        
        attn_out = self.bn_proj(self.proj(attn_out))
        
        local_out = self.local_conv(x)
        
        return self.act(attn_out + local_out + residual)

class SpaceToDepth(nn.Module):
    def __init__(self, c1, c2=None, block_size=2, *args, **kwargs):
        super().__init__()
        self.block_size = block_size
        c2 = c2 if c2 is not None else c1 * block_size * block_size
        self.c2 = c2
        expanded = c1 * block_size * block_size
        self.conv = nn.Conv2d(expanded, c2, 1, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        bs = self.block_size
        N, C, H, W = x.shape
        x = x.view(N, C, H // bs, bs, W // bs, bs)
        x = x.permute(0, 1, 3, 5, 2, 4).contiguous()
        x = x.view(N, C * bs * bs, H // bs, W // bs)
        return self.act(self.bn(self.conv(x)))
    
for name, module_class in [('EfficientViT_R8', EfficientViT_R8), ('SpaceToDepth', SpaceToDepth)]:
    setattr(ultralytics.nn.tasks, name, module_class)
    setattr(ul_modules, name, module_class)
    ultralytics.nn.tasks.__dict__[name] = module_class
    ul_modules.__dict__[name] = module_class
print("Registered: EfficientViT_R8 (Memory-Efficient), SpaceToDepth")

data_path = '/kaggle/input/datasets/ramgolla17042006/cplid-updated'
classes = ["insulator", "defect"]
nc = len(classes)
print(f"Classes found ({nc}): {classes}")

class_weights = [1.0] * nc
for i, name in enumerate(classes):
    lower = name.lower()
    if 'pollution' in lower or 'flashover' in lower or 'broken' in lower:
        class_weights[i] = 10.0
        print(f"Weight 10.0 → {name}")

data_yaml_dict = {
    'path': data_path,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test' if os.path.exists(os.path.join(data_path, 'images/test')) else 'images/val',
    'nc': nc,
    'names': classes
}
with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml_dict, f, default_flow_style=False)
print("data.yaml created")

def patched_parse_model(d, ch, verbose=True):
    from ultralytics.utils import LOGGER
    from ultralytics.nn.modules import (
        AIFI, C1, C2, C3, C3TR, OBB, SPP, SPPELAN, SPPF,
        Bottleneck, BottleneckCSP, C2f, C2fAttn, C3Ghost, C3k2,
        Classify, Concat, Conv, ConvTranspose, Detect, DWConv,
        DWConvTranspose2d, Focus, GhostBottleneck, GhostConv,
        HGBlock, HGStem, ImagePoolingAttn, Pose, RepC3, RepConv,
        RTDETRDecoder, Segment, WorldDetect
    )
    
    if isinstance(ch, int):
        ch = [ch]
    
    nc, act, scales = (d.get(x) for x in ("nc", "activation", "scales"))
    depth, width, max_channels = scales.get(d.get("scale", "n"), scales["n"])
    
    if act:
        Conv.default_act = eval(act)
    
    if verbose:
        LOGGER.info(f"\n{'':>3}{'from':>20}{'n':>3}{'params':>10}  {'module':<45}{'arguments':<30}")
    
    layers, save, c2 = [], [], ch[-1]
    
    for i, (f, n, m, args) in enumerate(d["backbone"] + d["head"]):
        args = list(args) if isinstance(args, (list, tuple)) else [args] if args else []
        
        if isinstance(m, str):
            if m == 'EfficientViT_R8':
                m = EfficientViT_R8
            elif m == 'SpaceToDepth':
                m = SpaceToDepth
            elif m in ('nn.Upsample', 'Upsample'):
                m = nn.Upsample
            else:
                m = getattr(ul_modules, m, None) or eval(m)
        
        n = max(round(n * depth), 1) if n > 1 else n
        
        if isinstance(f, int):
            c1 = ch[f]
        else:
            c1 = ch[f[0]] if len(f) == 1 else sum(ch[x] for x in f)
        
        if m in (Conv, ConvTranspose, GhostConv, Bottleneck, SPP, SPPF,
                 DWConv, Focus, BottleneckCSP, C1, C2, C2f, C3, C3TR,
                 C3Ghost, RepC3, C2fAttn, C3k2, RepConv, HGStem, HGBlock,
                 SPPELAN):
            c2 = args[0]
            if c2 != nc:
                c2 = min(c2, max_channels)
                c2 = int(c2 * width)
            args = [c1, c2, *args[1:]]
            
        elif m in (EfficientViT_R8, SpaceToDepth):
            if len(args) >= 1:
                c2 = args[0]
                if c2 != nc:
                    c2 = min(c2, max_channels)
                    c2 = int(c2 * width)
                args = [c1, c2, *args[1:]]
            else:
                c2 = c1
                args = [c1, c2]
                
        elif m is nn.Upsample:
            c2 = c1
            m_ = nn.Upsample(scale_factor=2, mode='nearest')
            t = "nn.Upsample"
            m_.np = 0
            m_.i, m_.f, m_.type = i, f, t
            if verbose:
                LOGGER.info(f"{i:>3}{str(f):>20}{n:>3}{0:10.0f}  {t:<45}{'scale_factor=2':<30}")
            save.extend(x % i for x in ([f] if isinstance(f, int) else f) if x != -1)
            layers.append(m_)
            if i == 0:
                ch = []
            ch.append(c2)
            continue
            
        elif m is Concat:
            c2 = sum(ch[x] for x in f)
            
        elif m in (Detect, Segment, Pose, OBB, WorldDetect):
            ch_list = [ch[x] for x in f]
            m_ = m(nc=nc, ch=ch_list)
            t = str(m)[8:-2].replace("__main__.", "")
            m_.np = sum(x.numel() for x in m_.parameters())
            m_.i, m_.f, m_.type = i, f, t
            if verbose:
                LOGGER.info(f"{i:>3}{str(f):>20}{n:>3}{m_.np:10.0f}  {t:<45}{'nc=%d, ch=%s' % (nc, ch_list):<30}")
            save.extend(x % i for x in ([f] if isinstance(f, int) else f) if x != -1)
            layers.append(m_)
            if i == 0:
                ch = []
            ch.append(c2)
            continue
            
        elif m is RTDETRDecoder:
            args.insert(0, [ch[x] for x in f])
        elif m is Classify:
            c2 = args[0]
            args = [c1, c2, *args[1:]]
        elif m is AIFI:
            c2 = args[0]
            if c2 != nc:
                c2 = min(c2, max_channels)
                c2 = int(c2 * width)
            args = [c1, c2, *args[1:]]
        else:
            c2 = c1
        
        m_ = nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)
        t = str(m)[8:-2].replace("__main__.", "")
        m_.np = sum(x.numel() for x in m_.parameters())
        m_.i, m_.f, m_.type = i, f, t
        
        if verbose:
            LOGGER.info(f"{i:>3}{str(f):>20}{n:>3}{m_.np:10.0f}  {t:<45}{str(args):<30}")
        
        save.extend(x % i for x in ([f] if isinstance(f, int) else f) if x != -1)
        layers.append(m_)
        
        if i == 0:
            ch = []
        ch.append(c2)
    
    return nn.Sequential(*layers), sorted(save)

ultralytics.nn.tasks.parse_model = patched_parse_model
print(" Patched parse_model")

custom_yaml=f"""
nc: {nc}
scales:
  n: [0.33, 0.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]          # 0
  - [-1, 1, Conv, [128, 3, 2]]         # 1
  - [-1, 3, C3k2, [128, True, 0.25]]   # 2
  - [-1, 1, EfficientViT_R8, [128]]    # 3

  - [-1, 1, Conv, [256, 3, 2]]         # 4
  - [-1, 6, C3k2, [256, True, 0.25]]   # 5

  - [-1, 1, Conv, [512, 3, 2]]         # 6
  - [-1, 6, C3k2, [512, True, 0.5]]    # 7

  - [-1, 1, Conv, [1024, 3, 2]]        # 8
  - [-1, 3, C3k2, [1024, True]]        # 9

  - [-1, 1, SPPF, [1024, 5]]           # 10

head:
  - [-1, 1, nn.Upsample, []]           # 11
  - [[-1, 7], 1, Concat, [1]]          # 12 (512 stage)
  - [-1, 3, C3k2, [512, False]]        # 13

  - [-1, 1, nn.Upsample, []]           # 14
  - [[-1, 5], 1, Concat, [1]]          # 15 (256 stage)
  - [-1, 3, C3k2, [256, False]]        # 16

  - [-1, 1, Conv, [256, 3, 2]]         # 17
  - [[-1, 13], 1, Concat, [1]]         # 18
  - [-1, 3, C3k2, [512, False]]        # 19

  - [-1, 1, Conv, [512, 3, 2]]         # 20
  - [[-1, 10], 1, Concat, [1]]         # 21
  - [-1, 3, C3k2, [1024, False]]       # 22

  - [[16, 19, 22], 1, Detect, []]      # 23
"""

with open('custom_efficientvit_r8_yolo.yaml', 'w') as f:
    f.write(custom_yaml)
print(" YAML saved")

print("\n" + "="*60)
print("Building YOLOv11 + EfficientViT_R8 (Memory-Efficient)...")
print("="*60)

model = YOLO('custom_efficientvit_r8_yolo.yaml')
print("\n Model built!")

model = model.load('yolo11n.pt')
print(" Weights loaded!")

model.info()

# Show R8 modules
print("\n" + "-"*50)
print("EfficientViT_R8 layers:")
for i, m in enumerate(model.model.model):
    if 'EfficientViT_R8' in type(m).__name__:
        print(f"  Layer {i}: c1={m.c1}, c2={m.c2}, heads={m.num_heads}")
print("-"*50)

class WeightedDetectionLoss(v8DetectionLoss):
    def __init__(self, model):
        super().__init__(model)
        device = next(model.parameters()).device
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(class_weights, device=device, dtype=torch.float),
            reduction='mean'
        )

model.model.criterion = WeightedDetectionLoss(model.model)
print(" Weighted loss")

original_bbox_iou = ultralytics.utils.metrics.bbox_iou
def siou_wrapper(*args, **kwargs):
    kwargs['SIoU'] = True
    kwargs.pop('CIoU', None)
    return original_bbox_iou(*args, **kwargs)
ultralytics.utils.metrics.bbox_iou = siou_wrapper
print("SIoU enabled")

previous_best = '/kaggle/working/runs/detect/insdef_yolo11_final_abliation_varient_3/weights/best.pt'

print("\n" + "="*60)
print("TRAINING (Memory-Efficient)")
print("="*60)

torch.cuda.empty_cache()

if os.path.exists(previous_best):
    print(f"Resuming from: {previous_best}")
    results = model.train(
        resume=previous_best,
        data='data.yaml',
        epochs=100,
        imgsz=640,
        batch=8,
        device=0,
        patience=50,
        optimizer='auto',
        plots=True,
        save=True,
        name='insdef_efficientvit_r8',
        cls=1.5,
        box=7.5
    )
else:
    print("Fresh training...")
    results = model.train(
        data='data.yaml',
        epochs=100,
        imgsz=640,
        batch=8,  
        device=0,
        patience=50,
        optimizer='auto',
        plots=True,
        save=True,
        name='insdef_efficientvit_r8',
        cls=1.5,
        box=7.5,
        pretrained=True
    )


test_results = model.val(data='data.yaml', split='test', plots=True)

print("\n" + "="*60)
print("RESULTS - YOLOv11 + EfficientViT_R8")
print("="*60)
print(f"  mAP@0.5      : {test_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {test_results.box.map:.4f}")
print(f"  Precision    : {test_results.box.p.mean():.4f}")
print(f"  Recall       : {test_results.box.r.mean():.4f}")
print("="*60)